# Land Cover Mapping Methods Comparison

## Pre-requisistes
- Earth Engine Initialization
- AOI retrieval

In [ ]:
import ee
import luma_ge
ee.Authenticate() #force=True use for re-authentication
ee.Initialize()

In [2]:
#############################  Area of Interest  ###########################
def get_aoi_from_gaul(country="Indonesia", province="Sumatera Selatan"):
    """
    Get Area of Interest geometry from GAUL administrative boundaries.
    
    Parameters:
    -----------
    country : str
        Country name (default: "Indonesia")
    province : str
        Province/state name (default: "Sumatera Selatan")
        
    Returns:
    --------
    ee.Geometry : Area of interest geometry
    """
    admin = ee.FeatureCollection("FAO/GAUL/2015/level1")
    aoi_fc = admin.filter(ee.Filter.eq('ADM0_NAME', country)).filter(
        ee.Filter.eq('ADM1_NAME', province)
    )
    return aoi_fc.geometry()
aoi = get_aoi_from_gaul()


## Direct Classification
1. Search The Imagery 
2. Define the predictor
3. Run the classification

In [ ]:
#import the library
import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image
aoi = geemap.shp_to_ee('C:/Data Spasial/Adaptive workflow testing/GPS-points/AOI_Oganilir.shp')
#initilize reflectance data retrieval class
reflectance = Reflectance_Data()
#retrieve the image collection
#use 2016-2017 image for better coverage
img_col, stat = reflectance.get_optical_data(aoi, #Aoi
                                       '2016-01-01', '2017-12-31', #start, end
                                       optical_data='L8_SR', #sensor
                                       cloud_cover=40, #cloud cover
                                       compute_detailed_stats=False)
#get the thermal band
thermal, stat = reflectance.get_thermal_bands(aoi, 
                                              '2016-01-01', '2017-12-31',
                                              thermal_data='L8_TOA',
                                              cloud_cover=40,
                                              compute_detailed_stats=False)
#initilize the compositing class
comp = final_Image()
#create the composite for multispectral and thermal data
med_landsat = comp.get_temporal_composite(img_col, aoi, reducer='Median')
median_landsat = med_landsat.select(
    med_landsat.bandNames().remove('AEROSOL')
)
thermal_median = comp.get_temporal_composite(thermal, aoi )
#define the visulization parameter and show them on the map
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
map = geemap.Map()
map.centerObject(aoi, zoom=10)
map.addLayer(thermal_median, {}, "Thermal")
map.addLayer(img_col,l8_sr_visparam, "Collection")
map.addLayer(median_landsat, l8_sr_visparam, "Median Image")
map

2026-04-14 17:29:27,686 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-04-14 17:29:27,687 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-04-14 17:29:27,687 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-04-14 17:29:27,687 - Reflectance_Data - INFO - Cloud cover threshold: 65%
2026-04-14 17:29:27,688 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-04-14 17:29:27,688 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-04-14 17:29:27,689 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-04-14 17:29:27,692 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-04-14 17:29:27,692 - Reflectance_Data - INFO - Starting thermal data fetch for Landsat 8 Top-of-atmosphere reflectance
2026-04-14 17:29:27,693 - Reflectance_Data - INFO - Date range: 2016-01-01 to 2017-12-31
2026-04-14 1

Map(center=[-3.4152616959981676, 104.60534276364243], controls=(WidgetControl(options=['position', 'transparen…

## Terrain and Spectral Predictors


In [30]:
from luma_ge.predictor import terrain_calculator, SpectralCalculator
# Initialize predictor backends
terrain_calc = terrain_calculator()
spectral_calc = SpectralCalculator()

# Choose DEM source and compute terrain layers
dem_source = 'NASADEM'
elevation = terrain_calc.calculate_elevation(aoi, dem_source=dem_source)
slope = terrain_calc.calculate_slope(aoi, dem_source=dem_source)
#aspect = terrain_calc.calculate_aspect(aoi, dem_source=dem_source)

# Visualize terrain layers on the map
terrain_vis = {
    'min': 0,
    'max': 3000,
    'palette': ['black', 'blue', 'green', 'yellow', 'red']
}
slope_vis = {
    'min': 0,
    'max': 45,
    'palette': ['white', 'lightblue', 'green', 'yellow', 'red']
}
aspect_vis = {
    'min': 0,
    'max': 360,
    'palette': ['white', 'blue', 'green', 'yellow', 'red', 'purple']
}

map.addLayer(elevation, terrain_vis, f"Elevation ({dem_source})")
map.addLayer(slope, slope_vis, f"Slope ({dem_source})")
#map.addLayer(aspect, aspect_vis, f"Aspect ({dem_source})")

# Compute spectral indices from the optical collection
indices_to_compute = ['NDVI', 'EVI', "MSAVI", "NDMI", "MNDWI", "DBSI"]
spectral_indices = spectral_calc.calculate_indices_with_collection(
    collection=img_col,
    aoi=aoi,
    index_list=indices_to_compute,
    reducer_method='median'
)

print('Computed terrain layers: elevation, slope, aspect')
print('Computed spectral indices:', spectral_indices.bandNames().getInfo() if spectral_indices else 'failed')

if spectral_indices:
    ndvi_vis = {'min': -1, 'max': 1, 'palette': ['purple', 'white', 'green']}
    evi_vis = {'min': -1, 'max': 1, 'palette': ['navy', 'white', 'lime']}
    map.addLayer(spectral_indices.select('NDVI'), ndvi_vis, 'NDVI')
    map.addLayer(spectral_indices.select('EVI'), evi_vis, 'EVI')
#stack the predictors and the imagery
predictor_stack = median_landsat.addBands(thermal_median).addBands(spectral_indices).addBands(elevation).addBands(slope).toFloat()

print('Final predictor stack bands:', predictor_stack.bandNames().getInfo() if predictor_stack else 'failed')

2026-04-14 15:22:23,131 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-04-14 15:22:23,132 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-04-14 15:22:23,133 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-04-14 15:22:23,134 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-04-14 15:22:23,135 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-04-14 15:22:23,135 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-04-14 15:22:23,137 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-04-14 15:22:23,138 - luma_ge.predictor - INFO - Successfully calculated slope layer using NASADEM DEM
2026-04-14 15:22:26,508 - luma_ge.predictor - INFO - Calculating spectral indices using map and median reducer on collection
2026-04-14 15:22:26,516 - luma_ge.predictor - INFO - Using default coefficie

Computed terrain layers: elevation, slope, aspect
Computed spectral indices: ['NDVI', 'EVI', 'MSAVI', 'NDMI', 'MNDWI', 'DBSI']
Final predictor stack bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'THERMAL', 'NDVI', 'EVI', 'MSAVI', 'NDMI', 'MNDWI', 'DBSI', 'elevation', 'slope']


## Classification 

In [31]:
#extract the pixel sample
from luma_ge.classification import FeatureExtraction, Generate_LULC
#samples
sample = geemap.shp_to_ee('C:/Data Spasial/Adaptive workflow testing/GPS-points/GT_2016/Ground_truth_2016_joint_epistem.shp')
features = FeatureExtraction()
train, test = features.stratified_split(sample, predictor_stack, 
                            class_prop='ID_epistem', train_ratio=0.80, seed=42)

2026-04-14 15:22:39,279 - pyogrio._io - INFO - Created 1,101 records


Stratified Random Split Training Pixel Size: 871
Stratified Random Split Testing Pixel Size: 230


In [32]:
#initilizae the classification class
clf = Generate_LULC()
#applied hard classification/original MS
classification_raw, model_raw = clf.hard_classification(training_data = train, #Ms only training data
 class_property='ID_epistem', 
 image=predictor_stack, 
 ntrees=300,
 return_model=True)
#Land cover class definition
lc_class = {
    1: {"name": "Secondary Dryland Forest",   "color": "#054504"},
    6: {"name": "Secondary Swamp Forest",   "color": "#059486"},
    16: {"name": "Mixed/home Garden", "color": "#0deb50"},
    15: {"name": "Rubber Agroforest",    "color": "#839248"},
    14: {"name": "Coffee Agroforest",    "color": "#df980a"},
    7: {"name": "Plantation Forest",    "color": "#09a726"},
    9: {"name": "Oil Palm Monoculture",    "color": "#d9cc66"},
    8: {"name": "Rubber monoculture",    "color": "#414127"},
    11: {"name": "Coconut monoculture",    "color": "#d9e66c"},
    12: {"name": "Other monoculture",    "color": "#6aa66d"},
    17: {"name": "Paddy Field",    "color": "#b1eb03"},
    13: {"name": "Other Monoculture",    "color": "#f1d900"},
    18: {"name": "Grass or Savanna",    "color": "#bdf2c0"},    
    21: {"name": "Cleared land",    "color": "#413d2f"},
    20: {"name": "Settlement",    "color": "#e00c0c"},
    24: {"name": "Fish Pond",    "color": "#e18adb"},
    23: {"name": "Water body",    "color": "#0b3bdb"},                  
}

In [33]:
#Evaluate model performance
print("Evaluating raw classification.")
try:
    model_acc_first = clf.evaluate_model(
        trained_model=model_raw,
        test_data=test,
        class_property='ID_epistem'
    )
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    
import pandas as pd
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {model_acc_first['overall_accuracy']:.4f} ({model_acc_first['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {model_acc_first['kappa']:.4f}")
print(f"Overall G-Mean: {model_acc_first['overall_gmean']:.4f}")
print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_df = pd.DataFrame({
    'Precision': model_acc_first['precision'],
    'Recall': model_acc_first['recall'],
    'F1-Score': model_acc_first['f1_scores'],
    'G-Mean': model_acc_first['gmean_per_class']
})
#Round to 4 decimal places
metrics_df = metrics_df.round(4)
display(metrics_df)

Evaluating raw classification.
=== Model Performance Summary ===
Overall Accuracy: 0.4609 (46.09%)
Kappa Coefficient: 0.3584
Overall G-Mean: 0.3955

=== Per-Class Metrics ===


,Precision,Recall,F1-Score,G-Mean
0,0.0000,0.0000,0.0000,0.0000
1,1.0000,0.0909,0.1667,0.3015
2,0.0000,0.0000,0.0000,0.0000
3,0.0000,0.0000,0.0000,0.0000
4,0.0000,0.0000,0.0000,0.0000
5,0.0000,0.0000,0.0000,0.0000
6,1.0000,1.0000,1.0000,1.0000
7,0.5000,0.1111,0.1818,0.2357
8,0.4507,0.5926,0.5120,0.5168
9,0.4000,0.3810,0.3902,0.3904


Overall Accuracy: 0.4561 (45.61%)
Kappa Coefficient: 0.3600

In [19]:
#visualize the feature importance
import plotly.express as px
importance = clf.get_feature_importance(model_raw, training_data=train, class_property='ID_epistem')
fig = px.bar(
                importance,
                x='Importance',
                y='Band',
                orientation='h',
                title='Kanal mana yang paling penting?',
                color='Importance',
                color_continuous_scale='Viridis',
                text='Importance'
            )
            
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(
                yaxis={'categoryorder': 'total ascending'},
                height=max(400, len(importance) * 30),
                showlegend=False
            )
            
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Importance=%{marker.color}<br>Band=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': {'bdata': ('SKu/aZ4wlkDEguMmeY6SQLiuXhsVcp' ... '3nQo9AZUAxSL0ej0DuzSRa2vmOQA=='),
                                   'dtype': 'f8'},
                         'coloraxis': 'coloraxis',
                         'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'text': {'bdata': ('SKu/aZ4wlkDEguMmeY6SQLiuXhsVcp' ... '3nQo9AZUAxSL0ej0DuzSRa2vmOQA=='),
                       'dtype': 'f8'},
              'textposition': 'outside',
              'texttemplate': '%{text:.3f}',
              'type': 'bar',
              'x': {'bdata': ('SKu/aZ4wlkDEguMmeY6SQLiuXhsVcp' ... '3nQo9AZUAxSL0ej0DuzSRa2vmOQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['elevation', 'slope', 'THERMAL', 'BLUE', 'MNDWI', 'aspect', 'NIR',
                          'NDMI', 'DBSI', 'EVI', 'SWIR1', 'NDVI', 'SWIR2', 'GBNDVI', 'MSAVI',
                          'GREEN', 'RED'], dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'coloraxis': {'colorbar': {'title': {'text': 'Importance'}},
                             'colorscale': [[0.0, '#440154'], [0.1111111111111111,
                                            '#482878'], [0.2222222222222222,
                                            '#3e4989'], [0.3333333333333333,
                                            '#31688e'], [0.4444444444444444,
                                            '#26828e'], [0.5555555555555556,
                                            '#1f9e89'], [0.6666666666666666,
                                            '#35b779'], [0.7777777777777778,
                                            '#6ece58'], [0.8888888888888888,
                                            '#b5de2b'], [1.0, '#fde725']]},
               'height': 510,
               'legend': {'tracegroupgap': 0},
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Kanal mana yang paling penting?'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Importance'}},
               'yaxis': {'anchor': 'x',
                         'categoryorder': 'total ascending',
                         'domain': [0.0, 1.0],
                         'title': {'text': 'Band'}}}
})

In [34]:
class_ids = list(lc_class.keys())
vis_params = {
    "min": min(class_ids),
    "max": max(class_ids),
    "palette": [lc_class[i]["color"] for i in class_ids]
}

legend_dict = {
    lc_class[i]["name"]: lc_class[i]["color"]
    for i in class_ids
}
m = geemap.Map()
m.centerObject(aoi, 8)
m.addLayer(classification_raw, vis_params, "MS_Only_Classification")
m.add_legend(title="Land Cover", legend_dict=legend_dict)
m

Map(center=[-3.2210694545062024, 104.16355582426586], controls=(WidgetControl(options=['position', 'transparen…

In [35]:
def get_sat_embedding(aoi, start_year, end_year):
     """
    Retrieves the satellite embedding data
    for the specified time range and area of interest (AOI).
    Returns:
        ee.Image: A median-composited, clipped image.
    """
     dataset = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

     image = dataset\
        .filterDate(f'{start_year}-01-01', f'{end_year}-01-01') \
         .filterBounds(aoi)
     mosaicked = image.mosaic().clip(aoi)
     return mosaicked
#get the satellite embedding data
data_2020 = get_sat_embedding(aoi, 2017, 2018)

In [36]:
#applied hard classification/original MS
classification_embed, model_embed = clf.hard_classification(training_data = train, #Ms only training data
 class_property='ID_epistem', 
 image=predictor_stack, 
 ntrees=500,
 return_model=True)

In [37]:
#Evaluate model performance
print("Evaluating embedding classification.")
try:
    model_acc_embed = clf.evaluate_model(
        trained_model=model_embed,
        test_data=test,
        class_property='ID_epistem'
    )
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    
import pandas as pd
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {model_acc_embed['overall_accuracy']:.4f} ({model_acc_embed['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {model_acc_embed['kappa']:.4f}")
print(f"Overall G-Mean: {model_acc_embed['overall_gmean']:.4f}")
print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_df = pd.DataFrame({
    'Precision': model_acc_embed['precision'],
    'Recall': model_acc_embed['recall'],
    'F1-Score': model_acc_embed['f1_scores'],
    'G-Mean': model_acc_embed['gmean_per_class']
})
#Round to 4 decimal places
metrics_df = metrics_df.round(4)
display(metrics_df)

Evaluating embedding classification.
=== Model Performance Summary ===
Overall Accuracy: 0.4478 (44.78%)
Kappa Coefficient: 0.3427
Overall G-Mean: 0.3400

=== Per-Class Metrics ===


,Precision,Recall,F1-Score,G-Mean
0,0.0000,0.0000,0.0000,0.0000
1,0.5000,0.0909,0.1538,0.2132
2,0.0000,0.0000,0.0000,0.0000
3,0.0000,0.0000,0.0000,0.0000
4,0.0000,0.0000,0.0000,0.0000
5,0.0000,0.0000,0.0000,0.0000
6,1.0000,1.0000,1.0000,1.0000
7,0.5000,0.1111,0.1818,0.2357
8,0.4507,0.5926,0.5120,0.5168
9,0.4000,0.3810,0.3902,0.3904


In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=median_landsat,
    description='Median_Landsat_2017_Ogan_ilir',
    folder='Earth Engine',
    fileNamePrefix='Median_Landsat_2017_Ogan_ilir',
    scale=30,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)


In [ ]:
#element and properties proxies
#ESA World cover dataset, used several times
esa_lc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
#1. Tree presencen
tree_pres =esa_lc.eq(5).rename('Tree_presence')
#2. Tree cover
tree_cov = ee.ImageCollection("projects/sat-io/open-datasets/GFCC30TC").filterDate('2017-01-01', '2017-12-31').mosaic().clip(aoi)
#3. Tree height 
#tree_ht = ee.ImageCollection("projects/sat-io/open-datasets/GLAD/GEDI_V27").mosaic().unmask()
landmask = ee.Image("projects/glad/OceanMask").lte(1)
tree_ht = ee.Image('projects/glad/GLCLU2020/v2/LCLUC_2015').updateMask(landmask)

#4. Leaf area index. Note: Modis data are too coarse to be used, but at this time, no better alternative is available
lai = ee.ImageCollection('MODIS/061/MOD15A2H')\
                  .filterDate('2017-01-01', '2017-12-31')
lai = lai.mean().multiply(0.001)
#5. Tree temporal type 
GCI30 = ee.ImageCollection("projects/sat-io/open-datasets/GCI30")
#5. Shrub presence 
shrub_pres = esa_lc.eq(10).rename('Shrub_Presence')  # Assuming class 10 corresponds to shrub presence
#6. woodyleafpheonology and type
dataset = ee.ImageCollection('MODIS/061/MCD12Q1')
#7 herbs presence
herbs_pres =  esa_lc.eq(20).rename('Herbs_Presence')  # Assuming class 20 corresponds to herb presence
#8 grass presence
grass_pres = esa_lc.eq(30).rename('Grass_Presence')  #
#builup presence
built_pres = esa_lc.eq(40).rename('Built_Presence')  # Assuming class 40 corresponds to built-up presence




In [ ]:
#